# Phase 4: PointNet–GAT Background-Data Scaling Study on Kaggle GPU

**Repository:** `https://github.com/KamitKoul/anomaly-detector-gnn`  
**Objective:** Scientifically rigorous data-scaling study evaluating the impact of background sample scale (5k to 900k events) on anomaly discrimination and mass sculpting in the PointNet–GAT hybrid architecture.

### Experimental Controls Frozen:
- Architecture: `PointNetGATHybrid` (exactly 7,957 parameters)
- Scientific Objective: Pure kinematic reconstruction ($\\lambda_{\\text{MET}} = 0.0, \\lambda_{\\text{adv}} = 0.0$)
- Mini-Batch Size: Fixed at $B = 32$ with strict `drop_last=True`
- Disentangled Designs:
  * **Experiment A (Fixed Epochs):** 20 epochs across all tiers ($D_0$–$D_6$)
  * **Experiment B (Fixed Updates):** Exactly 10,000 updates ($320,000$ event presentations)
- Seeds: 42, 123, 999

## 1. Environment Verification & GPU Diagnostics

In [ ]:
!nvidia-smi
!pip install -q torch-geometric tables
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")


## 2. Dataset Auto-Discovery & Data Linking
Discovers the LHC Olympics 2020 background and signal files from Kaggle input mounts and links them to `data/`.

In [ ]:
import os, glob
!mkdir -p data

bg_candidates = glob.glob("/kaggle/input/**/events_LHCO2020_backgroundMC_Pythia.h5", recursive=True) + \
                glob.glob("/kaggle/input/**/*background*.h5", recursive=True)
sig_candidates = glob.glob("/kaggle/input/**/W_prime_signal.h5", recursive=True) + \
                 glob.glob("/kaggle/input/**/events_anomalydetection.h5", recursive=True)

if bg_candidates:
    os.system(f"ln -sf '{bg_candidates[0]}' data/events_LHCO2020_backgroundMC_Pythia.h5")
    print(f"[LINKED] Background: {bg_candidates[0]} -> data/events_LHCO2020_backgroundMC_Pythia.h5")
else:
    print("[WARN] No background HDF5 found in /kaggle/input; check dataset attachment")

if sig_candidates:
    os.system(f"ln -sf '{sig_candidates[0]}' data/W_prime_signal.h5")
    print(f"[LINKED] Signal: {sig_candidates[0]} -> data/W_prime_signal.h5")
else:
    print("[WARN] No signal HDF5 found in /kaggle/input; check dataset attachment")

!ls -lh data/

## 3. Verify Dataset Splits & Partition Manifests
Executes partition verification to ensure all 12 `.npy` partition index files exist and match SHA-256 manifests.

In [ ]:
!python src/data/scaling_split.py

## 4. Run Complete Automated Unit Test Suite
Executes all 72 unit tests to ensure mathematical, architectural, and data integrity.

In [ ]:
!python -m unittest discover -s tests -p "test_*.py"

## 5. Execute Full-Budget D0 Training & Frozen Test Evaluation
Executes Experiment A Tier D0 (5,000 background events) for the full 20 epochs (exactly 3,120 optimizer updates).
Then evaluates both the best and final checkpoints on the frozen test set (50,000 held-out background + 25,000 held-out signal events).


In [ ]:
# 1. Execute Full-Budget D0 Training (20 Epochs, 3,120 Optimizer Updates)
!python experiments/scaling/experiment_a_fixed_epochs.py --tier D0 --seed 42 --epochs 20

# 2. Evaluate Best Checkpoint on Frozen Test Set (50k BG + 25k Signal)
!python evaluation/evaluate_scaling_study.py \n    --checkpoint results/scaling/experiment_a/expA_d0_5k_s42/checkpoint_best.pt \n    --output-json results/scaling/experiment_a/expA_d0_5k_s42/eval_metrics_best.json

# 3. Evaluate Final Checkpoint on Frozen Test Set
!python evaluation/evaluate_scaling_study.py \n    --checkpoint results/scaling/experiment_a/expA_d0_5k_s42/checkpoint_final.pt \n    --output-json results/scaling/experiment_a/expA_d0_5k_s42/eval_metrics_final.json


## 6. Execute Scaling Pilot Verification Gate (6 Runs)
Verifies boundary tiers ($D_0, D_3, D_6$) on Seed 42 before scaling.

In [ ]:
!python scripts/run_scaling_pilot.py

## 7. Execute Full Scaling Matrix (42 Runs across D0–D6)
Executes the complete matrix of 42 runs across 7 dataset tiers and 3 random seeds.

In [ ]:
# To run a specific tier or experiment, customize the flags:
# e.g. !python scripts/run_scaling_matrix.py --experiment A --tier D0 --seed 42

# Run the complete matrix:
!python scripts/run_scaling_matrix.py --experiment both --tier all --seed all

## 8. Package Scaling Results Bundle for Download
Compresses all logs, checkpoints, manifests, metrics, and reports into a downloadable archive.

In [ ]:
import os
ARCHIVE = "/kaggle/working/scaling_results_bundle.tar.gz"
!tar -czvf {ARCHIVE} results/scaling/ reports/scaling/ data/splits/ configs/scaling/
if os.path.exists(ARCHIVE):
    print(f"\n[SUCCESS] Archive created: {ARCHIVE} ({round(os.path.getsize(ARCHIVE) / 1024**2, 2)} MB)")